# Explainable AI - Grad CAM

## Carico il modello e l’immagine che vogliamo analizzare

- L’immagine andrà trattata esattamente come sono trattate le immagini con la quale abbiamo addestrato la rete

In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image

# Carico il modello pre-trained (in mod. valutazione)
model = models.resnet50(pretrained=True)
model.eval()

# Preprocessing dell'immagine
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

image = Image.open('path_image.png')
input_image = transform(image).unsqeeze(0)

## Carico il modello e l’immagine che vogliamo analizzare

- L’immagine andrà trattata esattamente come sono trattate le immagini con la quale abbiamo addestrato la rete

In [ ]:
target_layer = model.layer4[2].conv3

## Salvo le attivazioni del layer scelto

- Devi catturare le attivazioni di questo layer mentre passi l'immagine attraverso il modello
- Per farlo, uso gli hook* di PyTorch
- In PyTorch, gli hook sono funzioni che possono essere "agganciate" a vari livelli di una rete neurale (come layer o gradienti) per eseguire operazioni personalizzate durante il forward pass (passaggio in avanti) o il backward pass (passaggio all'indietro, per il calcolo dei gradienti)

In [ ]:
activations = []

# Hook per catturare le attivazioni del layer target
def forward_hook(module, input, output):
  activations.append(output)

# Registro l'hook
hook = target_layer.register_forward_hook(forward_hook)

## Calcolare i gradienti della classe di interesse

- Dopo aver passato l'immagine nel modello, calcola i gradienti rispetto alla classe di interesse y
- dy/dA^k, dove A^k è la mappa di attivazione del layer conv. scelto

In [ ]:
# Passa l'immagine nel modello
output = model(input_image)

# Seleziona la classe di interesse
target_class = output.argmax(dim=1).item()

# Imposta il gradiente solo per la classe target
model.zero_grad()
output[:, target_class].backward()

# Ottieni il gradiente del layer target
gradients = target_layer.weight.grad

## Media dei gradienti e generazione della heatmap

- Media i gradienti e combinali con le attivazioni per creare la heatmap

In [ ]:
# Ottieni le attivazioni salvate dall'hook
activation_map = activations[0].squeeze().detach()

# Media i gradienti per ottenere i pesi dei filtri
gradients_mean = torch.mean(gradients, dim=[2, 3], keepdim=True)

# Pesa l'activation map con i gradienti mediati
weighted_activation = gradients_mean * activation_map

# Somma su tutti i filtri per ottenere la heatmap finale
heatmap = weighted_activation.sum(dim=0).squeeze()

## Post processing della heatmap

- Normalizza la heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Normalizza la heatmap tra 0 e 1
heatmap = torch.clamp(heatmap, min=0)
heatmap /= torch.max(heatmap)

# Converti la heatmap in numpy per visualizzarla
heatmap = heatmap.numpy()
plt.imshow(heatmap, cmap='jet')
plt.colorbar()
plt.show()